In [0]:
# Databricks Notebook: 00_Init_Schema
# 初始化目录结构：schema + 产品主数据表
# 用法：新工作区新建 Notebook，Language 选 Python，粘贴全部代码，只需运行一次
# 注意：新工作区 UC 已启用，默认 catalog 即当前工作区 catalog，无需手动指定

# =====================================================
# 1. 创建 5 个 schema（分层架构）
# =====================================================
for schema in ["source", "bronze", "silver", "gold", "quality"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    print(f"✅ schema 就绪: {schema}")

# =====================================================
# 2. 产品主数据 source.raw_products（静态主数据）
# =====================================================
spark.sql("""
CREATE OR REPLACE TABLE source.raw_products (
    product       STRING,
    technology    STRING,
    die_size_mm2  DOUBLE,
    wafer_size    INT,
    route_steps   INT
)
""")

products_data = [
    ("IGBT-650V",     "Power Device",       40.5, 150, 190),
    ("SiC-MOS-1200V", "3rd Gen Semiconductor", 12.3, 150, 230),
    ("BCD-PMIC",      "Power Management IC", 8.1, 150, 150),
    ("SBD-Schottky",  "Schottky Diode",      5.2, 150, 120),
    ("FRD-Fast",      "Fast Recovery Diode", 5.0, 150, 120),
]
products_df = spark.createDataFrame(
    products_data,
    "product STRING, technology STRING, die_size_mm2 DOUBLE, wafer_size INT, route_steps INT"
)
spark.sql("DELETE FROM source.raw_products")
products_df.write.mode("append").format("delta").saveAsTable("source.raw_products")
print(f"✅ source.raw_products: {products_df.count()} 个产品")

# =====================================================
# 3. 验证
# =====================================================
spark.sql("SHOW SCHEMAS").show()
print("🎉 初始化完成！下一步：先跑 10_Daily_Data_Generator 造数，再跑 01→04 管道")


✅ schema 就绪: source
✅ schema 就绪: bronze
✅ schema 就绪: silver
✅ schema 就绪: gold
✅ schema 就绪: quality
✅ source.raw_products: 5 个产品
+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|              gold|
|information_schema|
|           quality|
|            silver|
|            source|
+------------------+

🎉 初始化完成！下一步：先跑 10_Daily_Data_Generator 造数，再跑 01→04 管道
